# Carnage Phase 0 → Phase 1 on Kaggle
Enable **GPU T4 x2**, Internet, and add a Kaggle Secret named `WANDB_API_KEY`. Checkpoints are published to a private Kaggle dataset.

In [ ]:
import os, pathlib, subprocess, sys, time
from kaggle_secrets import UserSecretsClient
REPO = pathlib.Path('/kaggle/working/Carnage-V1')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/vfxjamer/Carnage-V1.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
os.environ['CARNAGE_REPO_ROOT'] = str(REPO)
try:
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    print('WANDB_API_KEY secret unavailable; omit --wandb or add the secret before training.')
subprocess.run(['bash', str(REPO / 'kaggle_setup.sh')], cwd=REPO, check=True)

In [ ]:
import kagglehub
identity = kagglehub.whoami()
username = identity.get('username') if isinstance(identity, dict) else getattr(identity, 'username', None)
if not username: raise RuntimeError(f'Could not determine Kaggle username from {identity!r}')
DATASET_HANDLE = os.environ.get('CARNAGE_DATASET_HANDLE', f'{username}/carnage-checkpoints')
CHECKPOINTS = pathlib.Path('/kaggle/working/carnage-checkpoints')
subprocess.run([sys.executable, str(REPO/'scripts/kaggle_checkpoint_sync.py'), 'restore', '--handle', DATASET_HANDLE, '--checkpoint-root', str(CHECKPOINTS)], check=True)
print('Dataset:', DATASET_HANDLE)

In [ ]:
BINARY = REPO / 'build-kaggle/Carnage'
subprocess.run(['ctest', '--test-dir', str(REPO/'build-kaggle'), '-R', 'CarnageCudaSplitTests', '--output-on-failure'], check=True)
def smoke(layout, games):
    root = pathlib.Path(f'/kaggle/working/smoke-{layout}-{games}')
    command = [str(BINARY), str(REPO/'collision_meshes'), '--device', 'cuda', '--device-layout', layout, '--games', str(games), '--resume', 'none', '--checkpoint-root', str(root), '--smoke-test']
    started = time.monotonic(); result = subprocess.run(command); elapsed = time.monotonic() - started
    if result.returncode: raise RuntimeError(f'{layout}/{games} smoke test failed')
    return elapsed
layout_times = {layout: smoke(layout, 128) for layout in ('single', 'split')}
DEVICE_LAYOUT = min(layout_times, key=layout_times.get)
worker_times = {games: smoke(DEVICE_LAYOUT, games) for games in (128, 256, 512)}
NUM_GAMES = min(worker_times, key=worker_times.get)
print('Validated layout timings:', layout_times, 'selected:', DEVICE_LAYOUT)
print('Worker timings:', worker_times, 'selected games:', NUM_GAMES)

In [ ]:
os.environ.setdefault('CARNAGE_WANDB_GROUP', 'Phase 0 to Phase 1')
os.environ.setdefault('CARNAGE_WANDB_RUN', 'carnage-v1-fresh')
train = [str(BINARY), str(REPO/'collision_meshes'), '--device', 'cuda', '--device-layout', DEVICE_LAYOUT, '--policy-cuda-device', '0', '--critic-cuda-device', '1', '--games', str(NUM_GAMES), '--checkpoint-root', str(CHECKPOINTS), '--resume', 'auto']
if os.environ.get('WANDB_API_KEY'): train += ['--wandb', os.environ.get('CARNAGE_WANDB_PROJECT', 'carnage-v1')]
command = [sys.executable, str(REPO/'scripts/kaggle_checkpoint_sync.py'), 'run', '--handle', DATASET_HANDLE, '--checkpoint-root', str(CHECKPOINTS), '--', *train]
raise SystemExit(subprocess.run(command).returncode)